# Project: Smart Onboarding

**What you will build:** an agent that triages new-hire onboarding surveys — reading the free-text comment, catching when it contradicts the star rating, and routing each case to the right team. It combines **structured output** (1.3), **routing** (0.4), and a **guardrail** (1.5) into one small, business-shaped build. Mirrors the n8n "Smart Onboarding" project.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/32b_project_onboarding.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
print("Ready:", MODEL_NAME)

## The problem

New hires fill in a survey: a 1–5 rating and a free-text comment. The trap: people rate generously ("I don't want to complain") while the comment reveals a real problem. A dropdown says 4/5; the text says "I still have no laptop." We want the **text to win**, and each flagged case routed to whoever can fix it (IT, the manager, HR).

In [ ]:
responses = [
    {"employee": "Ana",  "rating": 5, "comment": "Great first week — the team is really welcoming!"},
    {"employee": "Ben",  "rating": 4, "comment": "Fine overall, but I still don't have my laptop or building access."},
    {"employee": "Cara", "rating": 3, "comment": "My manager hasn't met with me once and I don't know what my goals are."},
]

## Structured, typed triage

We define exactly the shape we want back (1.3), and let the routing choice be a `Literal` so it can only ever be a valid team. The system prompt makes the key rule explicit: the comment outranks the rating.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Assessment(BaseModel):
    employee: str
    true_sentiment: Literal["positive", "mixed", "negative"]
    text_reveals_problem: bool = Field(description="True if the comment shows a real issue, even with a high rating")
    issues: list[str] = Field(description="Short phrases naming each problem, empty if none")
    route: Literal["all_good", "needs_it", "needs_manager", "needs_hr"]

triage = Agent(
    model,
    output_type=Assessment,
    retries=2,   # give the output_validator room to trigger a self-correction (see 1.5)
    system_prompt=(
        "You triage new-hire onboarding surveys. The free-text comment matters MORE than the numeric rating: "
        "if the text reveals a problem, flag it even when the rating is high. Route by problem type — "
        "needs_it (equipment/access), needs_manager (goals, 1:1s, direction), needs_hr (serious/sensitive concerns), "
        "or all_good when there is genuinely no issue."
    ),
)

## A guardrail: keep the routing self-consistent

A classic failure is flagging a problem but routing to `all_good`. A one-line `output_validator` (1.5) catches that and makes the agent fix itself.

In [ ]:
from pydantic_ai import ModelRetry

@triage.output_validator
def consistent(a: Assessment) -> Assessment:
    if a.text_reveals_problem and a.route == "all_good":
        raise ModelRetry("You flagged a problem but routed to all_good — pick the matching team.")
    if not a.text_reveals_problem and a.route != "all_good":
        raise ModelRetry("You routed to a team but flagged no problem — set text_reveals_problem or route all_good.")
    return a

## Run the triage + route

Assess each response, then dispatch on the (validated) `route`. The dispatch is plain Python — the routing pattern from 0.4.

In [ ]:
ACTIONS = {
    "all_good":     "no action — send a welcome note",
    "needs_it":     "open an IT ticket",
    "needs_manager":"nudge the manager to schedule a 1:1",
    "needs_hr":     "escalate to HR",
}

for r in responses:
    result = await triage.run(f"Employee: {r['employee']}\nRating: {r['rating']}/5\nComment: {r['comment']}")
    a = result.output
    flag = "⚠" if a.text_reveals_problem else " "
    print(f"{flag} {a.employee:5} rating {r['rating']}/5 -> {a.route:13} ({a.true_sentiment}) -> {ACTIONS[a.route]}")
    if a.issues:
        print(f"      issues: {', '.join(a.issues)}")

Ben rated 4/5 but got routed to `needs_it`, and Cara's vague 3/5 became `needs_manager` — the comment overrode the star rating exactly as intended. You combined three ideas from the course (typed output, a guardrail, and routing) into one thing that would actually be useful at a company.

## Eval it — with the cases that actually bite

Three responses prove it runs; they don't prove it's *right*. The danger in onboarding triage isn't the easy "everything's great" case — it's **contradiction**: a high rating hiding a real problem, a harmless-sounding logistics gripe, a sensitive issue that must reach HR and not IT. Build a dataset of exactly those traps and measure the route (1.6):

In [ ]:
# (comment, rating, expected route) — deliberately full of contradictions and edge cases
eval_cases = [
    ("Great week, no blockers.",                         5, "all_good"),
    ("Laptop still missing, but everyone is nice.",       4, "needs_it"),      # high rating hides a problem
    ("I'm not comfortable with my manager yet.",          3, "needs_manager"),
    ("I don't know who to ask about payroll.",            4, "needs_hr"),
    ("Badge doesn't work and I can't enter the office.",  2, "needs_it"),
    ("The team is welcoming, but I have no clear tasks.",  4, "needs_manager"),  # positive tone, real gap
    ("I need to discuss a private accommodation.",        3, "needs_hr"),        # sensitive -> HR, not IT
    ("Everything is okay, just tired.",                   4, "all_good"),
    ("I got a 5/5 experience but still can't access email.", 5, "needs_it"),
    ("My mentor missed every meeting this week.",         2, "needs_manager"),
    ("Setup done, docs clear, happy so far.",             5, "all_good"),
    ("Payroll deducted the wrong amount from my first check.", 3, "needs_hr"),
]

passed = 0
for comment, rating, expected in eval_cases:
    r = await triage.run(f"Employee: Test\nRating: {rating}/5\nComment: {comment}")
    got = r.output.route
    passed += got == expected
    print(f"[{'PASS' if got == expected else 'FAIL'}] want {expected:13s} got {got:13s} | {comment[:45]}")
print(f"\nScore: {passed}/{len(eval_cases)}")

Expect a few misses on the subtlest ones (a low rating with a needs_manager comment, or HR-vs-IT ambiguity) — that's the eval doing its job: it turns "seems fine" into a number you can improve against by sharpening the route descriptions or adding examples to the prompt. The traps that fail are your to-do list.

::::{dropdown} 🔧 HR escalation as durable HITL
:color: info

`needs_hr` is the route you'd least want to fire-and-forget. In a production LangGraph version, send it to an **`interrupt()`** node (2.4) that pauses before any message is sent and waits for a human HR reviewer — same shape as 2.4: the graph persists its state and resumes with `Command(resume=...)` after approval. A wrong `needs_it` ticket is an annoyance; a mishandled `needs_hr` case is a real-world harm, which is exactly the line where you trade automation for a human check.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Routes to `all_good` despite a problem | Rating outweighs text in the model's judgement | Strengthen the "comment outranks rating" rule; the validator forces a retry |
| Invents issues not in the text | Prompt invites speculation | Add "only flag problems stated in the comment" |
| Wrong team | Overlapping route definitions | Sharpen each route's description; add a couple of examples (appendix) |
::::

::::{dropdown} 🛠️ Extend it (challenges)
:color: secondary

1. **Add memory.** Feed prior responses from the same employee via `message_history` (1.4) so repeat problems escalate.
2. **Grow the eval.** Port the cases above into a `pydantic-evals` `Dataset` (1.6) with an LLM judge for the borderline HR-vs-manager routes, and track the score as a percentage as you tune the prompt.
3. **Make it real.** Replace the print with a real action — an IT-ticket API (a real POST, like `live_weather` in 1.2), or a Slack message to the manager. Gate `needs_hr` behind the `interrupt()` review above.
4. **Batch it.** Wrap the loop behind the FastAPI service (30) so HR can POST a CSV of responses and get routed results.
::::

**What's next:** in **32c**, you will move beyond a single live request and build Atlas: a coworker that pauses for approval, survives reconstruction, and prevents a duplicated external effect.